# 01 Preprocess
Builds the shared train / val / test split and the shared face-crop cache (32 uniformly sampled,
face-cropped 224x224 frames per ElderReact clip; resumable; kept as one tar on Drive in Colab).
For `MODEL = 'clustering'` it also extracts the MediaPipe blendshapes from the same crops.

In [1]:
# Colab: mount Drive and run the bootstrap written by 00_setup.ipynb (code + packages). Locally: nothing happens.
import sys
if "google.colab" in sys.modules:
    from google.colab import drive; drive.mount("/content/drive")
    exec(open("/content/drive/MyDrive/emotion_detection_bootstrap.py").read())

In [2]:
MODEL = "s2d"            # "s2d" | "maedfer" | "clustering" (the crop cache is shared by all three)
RUN_DIAGNOSTIC = False   # True: face-detection diagnostic on a sample of clips (detectors, crops, MediaPipe)
REDO = False             # True: rebuild even if the outputs exist
SMOKE_TEST = False

In [3]:
from common.env import init
from common import data, stages
cfg = init(MODEL, smoke_test=SMOKE_TEST)

MODEL = 's2d' | data root: C:\Users\lukas_rmwb0n8\Desktop\Project\Scientific-computing-project---Emotion-detection | device: cuda


## Split
Official train + dev pooled and re-split 85 / 15 (seed 42); official test untouched.

In [4]:
df = stages.prepare_split(cfg)
data.split_summary(df, cfg.emotions)

split file: C:\Users\lukas_rmwb0n8\Desktop\Project\Scientific-computing-project---Emotion-detection\splits\elderreact_pooled_seed42.csv


,clips,mean_labels,no_label,single,multi,anger,disgust,fear,happiness,sadness,surprise
split,,,,,,,,,,,
train,824,1.842233,47,308,469,224,183,111,475,145,380
val,146,1.842466,5,58,83,50,32,12,80,30,65
test,353,1.535411,40,151,162,76,63,29,187,66,121


## Face-crop cache
RetinaFace on every sampled frame, eye-levelled `deepface` crop (config `faces`).

In [5]:
stages.build_face_cache(cfg, df, redo=REDO)

cache C:\Users\lukas_rmwb0n8\Desktop\Project\Scientific-computing-project---Emotion-detection\cache\faces224_retinaface_deepface_f32_jpg: 74 clips already cached, 1249 to build
  [50/1249] 0.5 min, ~11.0 min left
  [100/1249] 0.9 min, ~10.1 min left
  [150/1249] 1.3 min, ~9.5 min left
  [200/1249] 1.7 min, ~9.0 min left
  [250/1249] 2.1 min, ~8.6 min left
  [300/1249] 2.6 min, ~8.1 min left
  [350/1249] 3.0 min, ~7.7 min left
  [400/1249] 3.4 min, ~7.3 min left
  [450/1249] 3.9 min, ~6.8 min left
  [500/1249] 4.3 min, ~6.4 min left
  [550/1249] 4.7 min, ~6.0 min left
  [600/1249] 5.1 min, ~5.5 min left
  [650/1249] 5.6 min, ~5.1 min left
  [700/1249] 6.0 min, ~4.7 min left
  [750/1249] 6.4 min, ~4.2 min left
  [800/1249] 6.8 min, ~3.8 min left
  [850/1249] 7.2 min, ~3.4 min left
  [900/1249] 7.6 min, ~3.0 min left
  [950/1249] 8.1 min, ~2.5 min left
  [1000/1249] 8.5 min, ~2.1 min left
  [1050/1249] 8.9 min, ~1.7 min left
  [1100/1249] 9.3 min, ~1.3 min left
  [1150/1249] 9.7 min, ~0.8

,value
clips,1299
status ok,1299
no face in clip (centre-crop fallback),0
unreadable,0
sampled frames without a face (filled from nearest),0.2 %


## Diagnostic (optional)
Detection rate, clips without a face, box jitter, MediaPipe on crops vs full frames.

In [6]:
if RUN_DIAGNOSTIC:
    display(stages.face_diagnostic(cfg, df))

## Blendshapes (clustering only)

In [7]:
if cfg.model == "clustering":
    stages.extract_blendshapes(cfg, df, redo=REDO)